In [1]:
import numpy as np
import pandas as pd

from IPython.display import display, HTML

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Content Based Filtering System ##
- use item features
    - genre of the movie
    - tags by the users
- also use user features(which is not available on this dataset,so we are moving on from this)
    - Age-Group(young,teen,old)
    - Location
    - Gender
    - Movies Watched
- Does NOT involve user-item interactions (unlike collaborative filtering).


In [2]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")
tags = pd.read_csv("tags.csv")

In [3]:
movies['genres'] = movies['genres'].fillna('')                              # Filling Missing Values with Empty String

# Convert genres into TF-IDF features (by default: CSR Format)               # CSR = Compressed Sparse Row
tfidf = TfidfVectorizer(stop_words='english')
csr_matrix = tfidf.fit_transform(movies['genres'])

print("CSR Matrix Shape:", csr_matrix.shape)

# Get feature names (all the words from genres)
feature_names = tfidf.get_feature_names_out()

print(feature_names)

CSR Matrix Shape: (9742, 23)
['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'fi' 'film' 'genres' 'horror' 'imax'
 'listed' 'musical' 'mystery' 'noir' 'romance' 'sci' 'thriller' 'war'
 'western']


In [4]:
print(csr_matrix)

# In this matrix called CSR(compressed sparse row) Matrix; most of the elements are zero
# Thus, only Values and their coordinates are stored
# Out of the total possible entries (9742 × 23 = 224,066), only 23,185 are non-zero.
    # → Sparsity ≈ 90% (very sparse).

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 23185 stored elements and shape (9742, 23)>
  Coords	Values
  (0, 1)	0.41684567364693936
  (0, 2)	0.5162254711770092
  (0, 3)	0.5048454681396087
  (0, 4)	0.26758647689140014
  (0, 8)	0.482990142708577
  (1, 1)	0.5123612074824269
  (1, 3)	0.6205251727456431
  (1, 8)	0.5936619434123594
  (2, 4)	0.5709154064399099
  (2, 18)	0.8210088907493954
  (3, 4)	0.5050154397005037
  (3, 18)	0.726240982959826
  (3, 7)	0.46640480307738325
  (4, 4)	1.0
  (5, 0)	0.5493281743985543
  (5, 5)	0.6359470441562757
  (5, 20)	0.5420423542868653
  (6, 4)	0.5709154064399099
  (6, 18)	0.8210088907493954
  (7, 1)	0.6366993258087036
  (7, 3)	0.7711121633813997
  (8, 0)	1.0
  (9, 1)	0.6295217016667962
  (9, 0)	0.5530653284926609
  (9, 20)	0.5457299419583338
  :	:
  (9731, 0)	0.41272965170024634
  (9731, 19)	0.508925697730817
  (9731, 9)	0.508925697730817
  (9732, 2)	0.5502833875552382
  (9732, 4)	0.28524046407869114
  (9732, 0)	0.39038039438445316
  (9732,

In [5]:
# Convert the CSR matrix to DataFrame for easy viewing
tfidf_df = pd.DataFrame(csr_matrix.toarray(), 
                        index=movies['title'], 
                        columns=feature_names)

# Show first few movies with their TF-IDF genre weights
display(tfidf_df[0:10])

,action,adventure,animation,children,comedy,crime,documentary,drama,fantasy,fi,...,imax,listed,musical,mystery,noir,romance,sci,thriller,war,western
title,,,,,,,,,,,,,,,,,,,,,
Toy Story (1995),0.000000,0.416846,0.516225,0.504845,0.267586,0.000000,0.0,0.000000,0.482990,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
Jumanji (1995),0.000000,0.512361,0.000000,0.620525,0.000000,0.000000,0.0,0.000000,0.593662,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
Grumpier Old Men (1995),0.000000,0.000000,0.000000,0.000000,0.570915,0.000000,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.821009,0.0,0.000000,0.0,0.0
Waiting to Exhale (1995),0.000000,0.000000,0.000000,0.000000,0.505015,0.000000,0.0,0.466405,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.726241,0.0,0.000000,0.0,0.0
Father of the Bride Part II (1995),0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
Heat (1995),0.549328,0.000000,0.000000,0.000000,0.000000,0.635947,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.542042,0.0,0.0
Sabrina (1995),0.000000,0.000000,0.000000,0.000000,0.570915,0.000000,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.821009,0.0,0.000000,0.0,0.0
Tom and Huck (1995),0.000000,0.636699,0.000000,0.771112,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0
Sudden Death (1995),1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0


### Cosine Similarity ###
- **Measures similarity between two vectors by comparing the angle between them**.

- Ranges from 0 to 1 --->
    - 0 : completely different (orthogonal)
    - 1 : identical(same direction)
    - Negative values are possible if vectors can be negative (but with TF-IDF they aren’t).

- cosine_similarity(A,B):
    - finds cosine similarity between movie A & movie B.

- when we call cosine_similarity(tfidf_matrix, tfidf_matrix);
    - it goes like matrix multiplication and creates (n_m * n_m) matrix. [n_m = no. of movies]
    - in this matrix, all diagonal elements equals to 1 (Beacause similarity of movie to the same movie is Perfect 1).
    - As like the matrix multiplication occurs,it creates the cosine similarity matrix
    - this cosine matrix is symetric beacause, cosine_sim(A,B)=cosine_sim(B,A)

- It does NOT measure magnitude (length), only direction.

- Two documents with very different word counts can still be highly similar if they use the same terms.

In [6]:
# Compute cosine similarity between all movies (n_m * n_m)
cosine_sim = cosine_similarity(csr_matrix , csr_matrix)

print("Cosine Similarity Matrix Shape:", cosine_sim.shape)

Cosine Similarity Matrix Shape: (9742, 9742)


In [7]:
# Reverse mapping of movie titles to dataframe index
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()


def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]  # find row index of movie

    # Similarity scores for this movie with all others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort movies by similarity score (highest first)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Take top 10 similar movies (skip the first, it's itself)
    sim_scores = sim_scores[1:11]

    # Extract the indices of these movies
    movie_indices = [i[0] for i in sim_scores]

    # Return their titles + genres
    return movies[['movieId', 'title', 'genres']].iloc[movie_indices]


In [8]:
# Printing Genre of the movie
movie_name = "Jumanji (1995)"
print(movies.loc[movies["title"] == movie_name, "genres"].values[0])

Adventure|Children|Fantasy


In [9]:
title = "<h3>Recommendations for {movie_name} on the basis of genres:</h3>"
display(HTML(title))
display(get_recommendations(movie_name))

,movieId,title,genres
53,60,"Indian in the Cupboard, The (1995)",Adventure|Children|Fantasy
109,126,"NeverEnding Story III, The (1994)",Adventure|Children|Fantasy
767,1009,Escape to Witch Mountain (1975),Adventure|Children|Fantasy
1514,2043,Darby O'Gill and the Little People (1959),Adventure|Children|Fantasy
1556,2093,Return to Oz (1985),Adventure|Children|Fantasy
1617,2161,"NeverEnding Story, The (1984)",Adventure|Children|Fantasy
1618,2162,"NeverEnding Story II: The Next Chapter, The (1...",Adventure|Children|Fantasy
1799,2399,Santa Claus: The Movie (1985),Adventure|Children|Fantasy
3574,4896,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure|Children|Fantasy
6075,41566,"Chronicles of Narnia: The Lion, the Witch and ...",Adventure|Children|Fantasy


## Extended Content Based Filtering with tags Data ##

In [10]:
# Load tags dataset
tags = pd.read_csv("tags.csv")

# Keep only needed columns
tags = tags[["movieId", "tag"]]

print(tags.head())

   movieId              tag
0    60756            funny
1    60756  Highly quotable
2    60756     will ferrell
3    89774     Boxing story
4    89774              MMA


In [11]:
# Merge tags with movies on movieId
movies_tags = movies.merge(tags, on="movieId", how="left")

# Group all tags for each movie into one string
movies_tags = movies_tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x.dropna())).reset_index()

# Merge back into movies dataset
movies = movies.merge(movies_tags, on="movieId", how="left")

# Replace NaN with empty string
movies["tag"] = movies["tag"].fillna("")

display(movies[["title", "genres", "tag"]].head(10))


,title,genres,tag
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun
1,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game
2,Grumpier Old Men (1995),Comedy|Romance,moldy old
3,Waiting to Exhale (1995),Comedy|Drama|Romance,
4,Father of the Bride Part II (1995),Comedy,pregnancy remake
5,Heat (1995),Action|Crime|Thriller,
6,Sabrina (1995),Comedy|Romance,remake
7,Tom and Huck (1995),Adventure|Children,
8,Sudden Death (1995),Action,
9,GoldenEye (1995),Action|Adventure|Thriller,


In [12]:
# Combine Features : Here, "tags" are repeated 3 times → effectively giving them 3× weight.
movies["content"] = movies["genres"] + " " + (movies["tag"] + " ") * 3

In [13]:
# TF-IDF on combined content
tfidf = TfidfVectorizer(stop_words="english")
CSR_matrix = tfidf.fit_transform(movies["content"])

print("CSR Matrix Shape:", CSR_matrix.shape)

# Get feature names (all the words from genres)
feature_names = tfidf.get_feature_names_out()

print(feature_names)

CSR Matrix Shape: (9742, 1677)
['06' '1900s' '1920s' ... 'zombie' 'zombies' 'zooey']


In [14]:
cosine_sim = cosine_similarity(CSR_matrix, CSR_matrix)

In [15]:
# Reset index for easy lookup
movies = movies.reset_index()

def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]  # find row index of movie

    # Similarity scores for this movie with all others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort movies by similarity score (highest first)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Take top 10 similar movies (skip the first, it's itself)
    sim_scores = sim_scores[1:11]

    # Extract the indices of these movies
    movie_indices = [i[0] for i in sim_scores]

    # Return their titles + genres
    return movies[['movieId', 'title', 'content']].iloc[movie_indices]


In [16]:
# Printing content of the movie
movie_name = "Jumanji (1995)"
print(movies.loc[movies["title"] == movie_name, "content"].values[0])


Adventure|Children|Fantasy fantasy magic board game Robin Williams game fantasy magic board game Robin Williams game fantasy magic board game Robin Williams game 


In [17]:
title = f"<h3>Recommendations for {movie_name} on the basis of genres and tags:</h3>"
display(HTML(title))
display(get_recommendations(movie_name))

,movieId,title,content
6254,46972,Night at the Museum (2006),Action|Comedy|Fantasy|IMAX Ben Stiller Robin W...
9692,184471,Tomb Raider (2018),Action|Adventure|Fantasy adventure Alicia Vika...
4076,5816,Harry Potter and the Chamber of Secrets (2002),Adventure|Fantasy Magic Wizards Magic Wizards ...
5166,8368,Harry Potter and the Prisoner of Azkaban (2004),Adventure|Fantasy|IMAX Magic Wizards Magic Wiz...
6062,40815,Harry Potter and the Goblet of Fire (2005),Adventure|Fantasy|Thriller|IMAX Magic Wizards ...
744,971,Cat on a Hot Tin Roof (1958),Drama Tennessee Williams Tennessee Williams Te...
841,1104,"Streetcar Named Desire, A (1951)",Drama Tennessee Williams Tennessee Williams Te...
3065,4113,"Glass Menagerie, The (1987)",Drama Tennessee Williams Tennessee Williams Te...
3839,5388,Insomnia (2002),Action|Crime|Drama|Mystery|Thriller Al Pacino ...
3574,4896,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure|Children|Fantasy Everything you want...
